In [2]:
import torch


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.4.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/traitlets/config/application.py", line 1075, in launch_instance
    app.start()
  File "/Users/sanaurooj/Library/Python/3.12/lib/python/site-packages/ipykernel/kernelapp.py", line 739, in start
    self.io_lo

In [45]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt

In [6]:
import torch.nn as nn 
import torch.optim as optim 

## XOR classifier problem

In [94]:
X = torch.tensor([[0,0],
                 [0,1],
                 [1,0],
                 [1,1]],dtype=torch.float32)

y = torch.tensor([[0],
                  [1],
                  [1],
                  [0]],dtype=torch.float32)

print(f"x: {X}")
print(f"y: {y}")
print(f"y shape: {y.shape}")

x: tensor([[0., 0.],
        [0., 1.],
        [1., 0.],
        [1., 1.]])
y: tensor([[0.],
        [1.],
        [1.],
        [0.]])
y shape: torch.Size([4, 1])


In [95]:
class XORModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(2, 4)   # 2 features and 4 hidden layers
        self.fc2 = nn.Linear(4, 1)   # 4 hidden layers and 1 output layer

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))

        return x

In [96]:
model = XORModel()
model(X)

tensor([[0.6246],
        [0.6141],
        [0.6055],
        [0.6127]], grad_fn=<SigmoidBackward0>)

In [97]:
model(X).shape

torch.Size([4, 1])

In [98]:
model.fc1.bias

Parameter containing:
tensor([ 0.6697,  0.2333, -0.3688,  0.4866], requires_grad=True)

In [99]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)

In [100]:
for epoch in range(3000):
    #1.forward pass
    prediction = model(X)

    #2. loss calculation
    loss = criterion(prediction, y)

    #3. zero gradients
    optimizer.zero_grad()

    #4. backward propagation
    loss.backward()

    #5. update weights
    optimizer.step()

    if(epoch+1)%500==0:
        print(f"Epoch: {epoch+1} | Loss: {loss:.10f}")

print("\n── Results ──")
model.eval()
with torch.no_grad():
    raw = model(X)
    predicted = (raw >= 0.5).float()   # threshold at 0.5

    print(f"X shape: {X.shape}")
    print(f"raw shape: {raw.shape}")
    for i in range(4):
        inp  = X[i].int().tolist()
        x1 = int(X[i][0].item())           
        x2 = int(X[i][1].item()) 
        prob = raw[i].item()
        pred = int(predicted[i].item())
        true = int(y[i].item())
        tick = "✓" if pred == true else "✗"
        print(f"  {x1} XOR {x2} → prob={prob:.3f}  pred={pred}  true={true}  {tick}")

    acc = (predicted == y).float().mean().item()
    print(f"\nAccuracy: {acc*100:.0f}%")

Epoch: 500 | Loss: 0.0013306214
Epoch: 1000 | Loss: 0.0004091040
Epoch: 1500 | Loss: 0.0001981050
Epoch: 2000 | Loss: 0.0001147229
Epoch: 2500 | Loss: 0.0000731333
Epoch: 3000 | Loss: 0.0000493195

── Results ──
X shape: torch.Size([4, 2])
raw shape: torch.Size([4, 1])
  0 XOR 0 → prob=0.000  pred=0  true=0  ✓
  0 XOR 1 → prob=1.000  pred=1  true=1  ✓
  1 XOR 0 → prob=1.000  pred=1  true=1  ✓
  1 XOR 1 → prob=0.000  pred=0  true=0  ✓

Accuracy: 100%


In [40]:
!pip3 install torchinfo

Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: /Library/Frameworks/Python.framework/Versions/3.12/bin/python3.12 -m pip install --upgrade pip


In [41]:
from torchinfo import summary

summary(XORModel())

Layer (type:depth-idx)                   Param #
XORModel                                 --
├─Linear: 1-1                            12
├─Linear: 1-2                            5
Total params: 17
Trainable params: 17
Non-trainable params: 0

## Regression : Fit a Sine wave

In [102]:
X = torch.linspace(-3.14, 3.14, 1000).unsqueeze(1)
y = torch.sin(X) + 0.1 * torch.randn_like(X)

In [103]:
y.shape

torch.Size([1000, 1])

In [104]:
class SineModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(1, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [105]:
model = SineModel()

In [106]:
model(X)

tensor([[-1.5431e-01],
        [-1.5367e-01],
        [-1.5304e-01],
        [-1.5240e-01],
        [-1.5176e-01],
        [-1.5112e-01],
        [-1.5048e-01],
        [-1.4984e-01],
        [-1.4921e-01],
        [-1.4857e-01],
        [-1.4793e-01],
        [-1.4729e-01],
        [-1.4665e-01],
        [-1.4602e-01],
        [-1.4538e-01],
        [-1.4474e-01],
        [-1.4410e-01],
        [-1.4346e-01],
        [-1.4282e-01],
        [-1.4222e-01],
        [-1.4162e-01],
        [-1.4103e-01],
        [-1.4044e-01],
        [-1.3985e-01],
        [-1.3925e-01],
        [-1.3866e-01],
        [-1.3807e-01],
        [-1.3748e-01],
        [-1.3688e-01],
        [-1.3629e-01],
        [-1.3570e-01],
        [-1.3511e-01],
        [-1.3451e-01],
        [-1.3392e-01],
        [-1.3333e-01],
        [-1.3274e-01],
        [-1.3215e-01],
        [-1.3155e-01],
        [-1.3096e-01],
        [-1.3037e-01],
        [-1.2978e-01],
        [-1.2918e-01],
        [-1.2859e-01],
        [-1

In [107]:
model.fc1.bias.shape

torch.Size([64])

In [108]:
model.parameters

<bound method Module.parameters of SineModel(
  (fc1): Linear(in_features=1, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=64, bias=True)
  (fc3): Linear(in_features=64, out_features=1, bias=True)
)>

In [109]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.001)
# scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=200, min_lr=1e-5, factor=0.5, verbose=True)

In [114]:
for epoch in range(2000):
    #1. forward pass
    prediction = model(X)

    #2. loss calculation 
    loss = criterion(prediction, y)

    #3. gradient zero
    optimizer.zero_grad()
    
    #4. backward pass
    loss.backward()

    #5. upgrade weights
    optimizer.step()

    # #6. learning rate scheduling
    # scheduler.step(loss)

    if((epoch+1)%500==0):
        print(f"Epoch: {epoch+1} | Loss: {loss:.10f} ")


Epoch: 500 | Loss: 0.0098187067 
Epoch: 1000 | Loss: 0.0098119164 
Epoch: 1500 | Loss: 0.0098318066 
Epoch: 2000 | Loss: 0.0098027559 


In [ ]:
summary(SineModel())

Layer (type:depth-idx)                   Param #
SineModel                                --
├─Linear: 1-1                            128
├─Linear: 1-2                            4,160
├─Linear: 1-3                            65
Total params: 4,353
Trainable params: 4,353
Non-trainable params: 0